# Install Dependencies

In [ ]:
!pip install -q emoji
!pip install -q PyArabic
!pip install -q arabert
!pip install -q openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 590.6/590.6 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.0/185.0 kB 4.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.9/250.9 kB 4.5 MB/s eta 0:00:00


# Import Required Modules

In [ ]:
import pandas as pd
import re
import pyarabic.araby as araby
import string
import emoji

from transformers import AutoTokenizer, AutoModel
import torch

# Load Collected climate

In [ ]:
climate = pd.read_excel('All Climate Change Data - All Related.xlsx')

climate.drop_duplicates(subset='text', inplace = True)
climate.dropna(inplace = True, subset='text')
climate.reset_index(drop=True, inplace = True)

climate.shape

(23959, 6)

# Preprocessing

In [ ]:
climate['text'] = climate['text'].str.replace(r'[^\w\s]+', ' ')
climate['text'] = climate['text'].str.replace("\s+", " ", regex=True)

climate.head()

,text,Label 1,Label 2,Label 4,Label 3,Final Label
0,هذه ال٢.٥٪ لا تنطلق إلى الفضاء الكوني فتحتبس و...,Negative,Positive,Negative,Neutral,Negative
1,#عاجل | إدارة الكوارث والطوارئ التركية: إزالة ...,Negative,Neutral,Neutral,Positive,Neutral
2,RT @USUN: عُقد في مالطا هذا الأسبوع أول اجتماع...,Positive,Neutral,Neutral,Negative,Neutral
3,رغم ارتفاع درجات الحرارة وأشعة الشمس اللاهبة و...,Neutral,Positive,Positive,Negative,Positive
4,قبل أيام تم استضافتي لتسجيل بودكاست بحكم اختصا...,Positive,Neutral,Neutral,Negative,Neutral


In [ ]:
arabic_punctuations = '''`÷×؛<>_()*&^%][ـ،/:"؟.,'{}~¦+|!”…“–ـ'''
english_punctuations = string.punctuation
punctuations_list = arabic_punctuations + english_punctuations

def normalize_arabic(text):
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("گ", "ك", text)
    return text

In [ ]:
climate['text'] = climate['text'].apply(normalize_arabic)

climate.head()

,text,Label 1,Label 2,Label 4,Label 3,Final Label
0,هذه ال٢.٥٪ لا تنطلق الى الفضاء الكوني فتحتبس و...,Negative,Positive,Negative,Neutral,Negative
1,#عاجل | ادارة الكوارث والطوارئ التركية: ازالة ...,Negative,Neutral,Neutral,Positive,Neutral
2,RT @USUN: عُقد في مالطا هذا الاسبوع اول اجتماع...,Positive,Neutral,Neutral,Negative,Neutral
3,رغم ارتفاع درجات الحرارة واشعة الشمس اللاهبة و...,Neutral,Positive,Positive,Negative,Positive
4,قبل ايام تم استضافتي لتسجيل بودكاست بحكم اختصا...,Positive,Neutral,Neutral,Negative,Neutral


In [ ]:
def data_cleaning(text):
    """Clean and preprocess text climate.
    Args:
        text (pd.Series): A pandas Series containing text climate to be cleaned.
    Returns:
        pd.Series: A pandas Series with the cleaned text climate.

    Cleaning Steps:
    - Removes emojis and special characters like '\x89Û_', '&amp', etc.
    - Replaces consecutive dots with an empty string.
    - Removes '#' symbol from text.
    - Removes user names starting with '@'.
    - Removes URLs starting with 'http' or 'https'.
    - Remove diacritics.
    - Remove English.
    - fix elongated words
    - Normalize Characters
    - Removes extra whitespaces between words.

    """
    clean = text
    # Replace consecutive dots with an empty string
    pattern = re.compile('\\.+?(?=\B|$)')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=''))
    # Replace '\x89Û_' with a whitespace
    pattern = re.compile('\x89Û_')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=' '))
    # Replace newline characters with a whitespace
    pattern = re.compile('\\n')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=' '))
    # Remove '#' symbol from text
    clean = clean.apply(lambda r: r.replace('#', ''))
    # Remove '_' symbol from text
    pattern = re.compile('_')
    clean = clean.apply(lambda r: re.sub(pattern, ' ', r))
    # Replace user names with '@'
    pattern = re.compile('@[a-zA-Z0-9\_]+')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl='@'))
    # Remove URLs
    pattern = re.compile('https?\S+(?=\s|$)')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl='www'))
    # Remove emojis
    clean = clean.apply(lambda r: emoji.replace_emoji(r, replace=""))
    # Remove diacritics
    clean = clean.apply(lambda r: araby.strip_diacritics(r))
    # Remove English
    # pattern = re.compile(r'[a-zA-Z]+')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=''))
    # Remove extra whitespaces
    clean = clean.apply(lambda r: ' '.join(r.split()))  # Remove extra whitespaces between words

    return clean

In [ ]:
climate['text'] = data_cleaning(climate['text'])

In [ ]:
def remove_ids(text):
  return text.split("—")[0].strip()

climate['text'] = climate['text'].apply(remove_ids)
climate.head()

,text,Label 1,Label 2,Label 4,Label 3,Final Label
0,هذه ال٢.٥٪ لا تنطلق الى الفضاء الكوني فتحتبس و...,Negative,Positive,Negative,Neutral,Negative
1,عاجل | ادارة الكوارث والطوارئ التركية: ازالة ا...,Negative,Neutral,Neutral,Positive,Neutral
2,RT @: عقد في مالطا هذا الاسبوع اول اجتماع لمجل...,Positive,Neutral,Neutral,Negative,Neutral
3,رغم ارتفاع درجات الحرارة واشعة الشمس اللاهبة و...,Neutral,Positive,Positive,Negative,Positive
4,قبل ايام تم استضافتي لتسجيل بودكاست بحكم اختصا...,Positive,Neutral,Neutral,Negative,Neutral


In [ ]:
climate.drop_duplicates(subset='text', inplace = True)
climate.dropna(inplace = True, subset='text',)
climate.shape

(23959, 6)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)
tfidf_matrix = vectorizer.fit_transform(climate['text'].tolist())

In [ ]:
embedding = pd.DataFrame(data = tfidf_matrix.toarray())
embedding.head()

,0,1,2,3,4,5,6,7,8,9,...,4990,4991,4992,4993,4994,4995,4996,4997,4998,4999
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
embedding['Label'] = climate['Final Label']

In [ ]:
embedding.to_csv('Climate_TFIDF.csv', index = False)

# Load ASAD Data

In [ ]:
asad = pd.read_csv('train_all.csv')

asad['Text'] = asad['Text'].str.replace(r'[^\w\s]+', '')
asad['Text'] = asad['Text'].str.replace("\s+", " ", regex=True)

asad['Text'] = asad['Text'].apply(normalize_arabic)

asad['Text'] = data_cleaning(asad['Text'])
asad['Text'] = asad['Text'].apply(remove_ids)

In [ ]:
asad.dropna(inplace = True)
asad = asad.drop_duplicates(subset='Text')
asad.shape

(53572, 3)

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000)
tfidf_matrix = vectorizer.fit_transform(asad['Text'].tolist())

In [ ]:
embedding = pd.DataFrame(data = tfidf_matrix.toarray())
embedding.head()

,0,1,2,3,4,5,6,7,8,9,...,4990,4991,4992,4993,4994,4995,4996,4997,4998,4999
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
embedding['Label'] = asad['sentiment'].str.title()

In [ ]:
embedding.to_csv('ASAD_TFIDF.csv', index = False)

# Load ASTD Data

In [ ]:
astd = pd.read_csv('Tweets.txt', delimiter='\t', header=None, names=['text', 'Label'])  # Change delimiter if needed
astd = astd[astd['Label']!='OBJ']
astd['Label'] = astd['Label'].map({
    'NEG': 'Negative',
    'NEUTRAL': 'Neutral',
    'POS': 'Positive'
})

astd['text'] = astd['text'].str.replace(r'[^\w\s]+', '')
astd['text'] = astd['text'].str.replace("\s+", " ", regex=True)

astd['text'] = astd['text'].apply(normalize_arabic)

astd['text'] = data_cleaning(astd['text'])
astd['text'] = astd['text'].apply(remove_ids)

astd.dropna(inplace = True)
astd = astd.drop_duplicates(subset = 'text')

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000)
tfidf_matrix = vectorizer.fit_transform(astd['text'].tolist())

In [ ]:
embedding = pd.DataFrame(data = tfidf_matrix.toarray())
embedding.head()

,0,1,2,3,4,5,6,7,8,9,...,4990,4991,4992,4993,4994,4995,4996,4997,4998,4999
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
embedding['Label'] = astd['Label']

In [ ]:
embedding.to_csv('ASTD_TFIDF.csv', index = False)

# Load SemEval Data

In [ ]:
semeval = pd.read_csv('SemEval2017-task4-train.subtask-A.arabic.txt', delimiter='\t', header=None, names=['id', 'sentiment', 'text'])
semeval = semeval[['text', 'sentiment']]

semeval['sentiment'] = semeval['sentiment'].map({
    'negative': 'Negative',
    'neutral': 'Neutral',
    'positive': 'Positive'
})


semeval['text'] = semeval['text'].str.replace(r'[^\w\s]+', '')
semeval['text'] = semeval['text'].str.replace("\s+", " ", regex=True)

semeval['text'] = semeval['text'].apply(normalize_arabic)

semeval['text'] = data_cleaning(semeval['text'])
semeval['text'] = semeval['text'].apply(remove_ids)

semeval.dropna(inplace = True)
semeval = semeval.drop_duplicates(subset = 'text')


In [ ]:
vectorizer = TfidfVectorizer(max_features=5000)
tfidf_matrix = vectorizer.fit_transform(semeval['text'].tolist())

In [ ]:
embedding = pd.DataFrame(data = tfidf_matrix.toarray())
embedding.head()

,0,1,2,3,4,5,6,7,8,9,...,4990,4991,4992,4993,4994,4995,4996,4997,4998,4999
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
embedding['Label'] = semeval['sentiment'].str.title()

In [ ]:
embedding.to_csv('SemEval_TFIDF.csv', index = False)

# Similarity between CLimate and ASTD

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

df1 = pd.read_csv('/content/Climate_TFIDF.csv')
df2 = pd.read_csv('/content/ASTD_TFIDF.csv')

neg1 = df1[df1['Label'] == 'Negative']
neg2 = df2[df2['Label'] == 'Negative']

# Convert embeddings to numpy arrays
embeddings1 = np.vstack(neg1.drop(columns={'Label'}, axis = 1).values)  # shape: (N1, D)
embeddings2 = np.vstack(neg2.drop(columns={'Label'}, axis = 1).values)  # shape: (N2, D)

all_similarities = []

for vec in embeddings1:
    sims = cosine_similarity([vec], embeddings2)  # shape: (1, N2)
    all_similarities.extend(sims[0])

average_similarity = np.mean(all_similarities)

print(f"Average cosine similarity between negative samples: {average_similarity:.3f}")

Average cosine similarity between negative samples: 0.003


In [5]:
pos1 = df1[df1['Label'] == 'Positive']
pos2 = df2[df2['Label'] == 'Positive']

# Convert embeddings to numpy arrays
embeddings1 = np.vstack(pos1.drop(columns={'Label'}, axis = 1).values)  # shape: (N1, D)
embeddings2 = np.vstack(pos2.drop(columns={'Label'}, axis = 1).values)  # shape: (N2, D)

all_similarities = []

for vec in embeddings1:
    sims = cosine_similarity([vec], embeddings2)  # shape: (1, N2)
    all_similarities.extend(sims[0])

average_similarity = np.mean(all_similarities)

print(f"Average cosine similarity between positive samples: {average_similarity: .3f}")

Average cosine similarity between positive samples: 0.003


In [6]:
neu1 = df1[df1['Label'] == 'Neutral']
neu2 = df2[df2['Label'] == 'Neutral']

# Convert embeddings to numpy arrays
embeddings1 = np.vstack(neu1.drop(columns={'Label'}, axis = 1).values)  # shape: (N1, D)
embeddings2 = np.vstack(neu2.drop(columns={'Label'}, axis = 1).values)  # shape: (N2, D)

all_similarities = []

for vec in embeddings1:
    sims = cosine_similarity([vec], embeddings2)  # shape: (1, N2)
    all_similarities.extend(sims[0])

average_similarity = np.mean(all_similarities)

print(f"Average cosine similarity between neutral samples: {average_similarity: .3f}")

Average cosine similarity between neutral samples: 0.003


In [7]:
# Convert embeddings to numpy arrays
embeddings1 = np.vstack(df1.drop(columns={'Label'}, axis = 1).values)  # shape: (N1, D)
embeddings2 = np.vstack(df2.drop(columns={'Label'}, axis = 1).values)  # shape: (N2, D)

all_similarities = []

for vec in embeddings1:
    sims = cosine_similarity([vec], embeddings2)  # shape: (1, N2)
    all_similarities.extend(sims[0])

average_similarity = np.mean(all_similarities)

print(f"Average cosine similarity between all samples: {average_similarity: .3f}")

Average cosine similarity between all samples: 0.003


# Similarity Between Climate and ASAD

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

df1 = pd.read_csv('/content/Climate_TFIDF.csv')
df2 = pd.read_csv('/content/ASAD_TFIDF.csv')

neg1 = df1[df1['Label'] == 'Negative']
neg2 = df2[df2['Label'] == 'Negative']

# Convert embeddings to numpy arrays
embeddings1 = np.vstack(neg1.drop(columns={'Label'}, axis = 1).values)  # shape: (N1, D)
embeddings2 = np.vstack(neg2.drop(columns={'Label'}, axis = 1).values)  # shape: (N2, D)

all_similarities = []

for vec in embeddings1:
    sims = cosine_similarity([vec], embeddings2)  # shape: (1, N2)
    all_similarities.extend(sims[0])

average_similarity = np.mean(all_similarities)

print("Average cosine similarity between negative samples:", average_similarity)

Average cosine similarity between negative samples: 0.0019740257083951172


In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

df1 = pd.read_csv('/content/Climate_TFIDF.csv')
df2 = pd.read_csv('/content/ASAD_TFIDF.csv')

pos1 = df1[df1['Label'] == 'Positive']
pos2 = df2[df2['Label'] == 'Positive']

# Convert embeddings to numpy arrays
embeddings1 = np.vstack(pos1.drop(columns={'Label'}, axis = 1).values)  # shape: (N1, D)
embeddings2 = np.vstack(pos2.drop(columns={'Label'}, axis = 1).values)  # shape: (N2, D)

all_similarities = []

for vec in embeddings1:
    sims = cosine_similarity([vec], embeddings2)  # shape: (1, N2)
    all_similarities.extend(sims[0])

average_similarity = np.mean(all_similarities)

print("Average cosine similarity between positive samples:", average_similarity)

Average cosine similarity between positive samples: 0.001870212090076539


In [ ]:
neu1 = df1[df1['Label'] == 'Neutral']
neu2 = df2[df2['Label'] == 'Neutral']

# Convert embeddings to numpy arrays
embeddings1 = np.vstack(neu1.drop(columns={'Label'}, axis = 1).values)  # shape: (N1, D)
embeddings2 = np.vstack(neu2.drop(columns={'Label'}, axis = 1).values)  # shape: (N2, D)

all_similarities = []

for vec in embeddings1:
    sims = cosine_similarity([vec], embeddings2)  # shape: (1, N2)
    all_similarities.extend(sims[0])

average_similarity = np.mean(all_similarities)

print("Average cosine similarity between neutral samples:", average_similarity)

Average cosine similarity between neutral samples: 0.001913755674751224


In [ ]:
# Convert embeddings to numpy arrays
embeddings1 = np.vstack(df1.drop(columns={'Label'}, axis = 1).values)  # shape: (N1, D)
embeddings2 = np.vstack(df2.drop(columns={'Label'}, axis = 1).values)  # shape: (N2, D)

all_similarities = []

for vec in embeddings1:
    sims = cosine_similarity([vec], embeddings2)  # shape: (1, N2)
    all_similarities.extend(sims[0])

average_similarity = np.mean(all_similarities)

print("Average cosine similarity between all samples:", average_similarity)

Average cosine similarity between all samples: 0.00191933115774096


# Similarity Between Climate and SemEval

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

df1 = pd.read_csv('/content/Climate_TFIDF.csv')
df2 = pd.read_csv('/content/SemEval_TFIDF.csv')

neg1 = df1[df1['Label'] == 'Negative']
neg2 = df2[df2['Label'] == 'Negative']

# Convert embeddings to numpy arrays
embeddings1 = np.vstack(neg1.drop(columns={'Label'}, axis = 1).values)  # shape: (N1, D)
embeddings2 = np.vstack(neg2.drop(columns={'Label'}, axis = 1).values)  # shape: (N2, D)

all_similarities = []

for vec in embeddings1:
    sims = cosine_similarity([vec], embeddings2)  # shape: (1, N2)
    all_similarities.extend(sims[0])

average_similarity = np.mean(all_similarities)

print("Average cosine similarity between negative samples:", average_similarity)

Average cosine similarity between negative samples: 0.002597571993779675


In [8]:
pos1 = df1[df1['Label'] == 'Positive']
pos2 = df2[df2['Label'] == 'Positive']

# Convert embeddings to numpy arrays
embeddings1 = np.vstack(pos1.drop(columns={'Label'}, axis = 1).values)  # shape: (N1, D)
embeddings2 = np.vstack(pos2.drop(columns={'Label'}, axis = 1).values)  # shape: (N2, D)

all_similarities = []

for vec in embeddings1:
    sims = cosine_similarity([vec], embeddings2)  # shape: (1, N2)
    all_similarities.extend(sims[0])

average_similarity = np.mean(all_similarities)

print(f"Average cosine similarity between positive samples: {average_similarity:.3f}")

Average cosine similarity between positive  samples: 0.003


In [10]:
neu1 = df1[df1['Label'] == 'Neutral']
neu2 = df2[df2['Label'] == 'Neutral']

# Convert embeddings to numpy arrays
embeddings1 = np.vstack(neu1.drop(columns={'Label'}, axis = 1).values)  # shape: (N1, D)
embeddings2 = np.vstack(neu2.drop(columns={'Label'}, axis = 1).values)  # shape: (N2, D)

all_similarities = []

for vec in embeddings1:
    sims = cosine_similarity([vec], embeddings2)  # shape: (1, N2)
    all_similarities.extend(sims[0])

average_similarity = np.mean(all_similarities)

print(f"Average cosine similarity between neutral samples: {average_similarity:.3f}")

Average cosine similarity between neutral samples: 0.003


In [11]:
print(f"Average cosine similarity between all samples: {average_similarity:.3f}")

Average cosine similarity between all samples: 0.003
